# UNCP 5 | Model and executed results

کد مدل و یک بررسی اجرایی در همین دفترچه اجرا شده‌اند. اعداد سه seed از فایل‌های نتیجهٔ آزمایش کامل که پیش‌تر اجرا شده‌اند، در سلول آخر ثبت و با میانگین همان فایل‌ها کنترل می‌شوند. این دفترچه آموزش کامل سه seed را دوباره اجرا نمی‌کند؛ برای بازآموزی، داده، کد benchmark و تنظیمات همان پروژه لازم است.

In [1]:
"""UNCP 5: validation-selected fusion of UNCP 2 and a training-only transition model.

Run after the three UNCP 2 grouped-user checkpoints exist. The test set is
accessed only after the fusion rule has been selected on validation data.
"""
import collections
import csv
import json
import math
from pathlib import Path

import numpy as np
import torch
# build_model is required only by main() during full training
# Full training needs the benchmark helpers from the accompanying project

HERE = Path.cwd()
SOURCE = HERE / 'results_uncp2'
OUT = HERE / 'results_adaptive_fusion'
SEEDS = (11, 23, 37)


def transition_tables(train, encoder):
    tables = [collections.defaultdict(collections.Counter) for _ in range(4)]
    global_counts = np.ones(len(encoder.outputs), dtype=np.float64) * .1
    for item in train:
        target = encoder.outputs[item['target']]
        global_counts[target] += 1
        context = item['context']
        for order in range(1, min(4, len(context)) + 1):
            tables[order-1][tuple(context[-order:])][target] += 1
    return tables, global_counts / global_counts.sum()


def transition_predictions(data, tables, prior):
    probs = np.empty((len(data), len(prior)), dtype=np.float64)
    supports = np.zeros(len(data), dtype=np.float64)
    for i, item in enumerate(data):
        context = item['context']
        counter = None
        for order in range(min(4, len(context)), 0, -1):
            found = tables[order-1].get(tuple(context[-order:]))
            if found:
                counter = found
                break
        if counter is None:
            probs[i] = prior
        else:
            support = sum(counter.values())
            supports[i] = support
            p = prior * 2.0
            for target, count in counter.items():
                p[target] += count
            probs[i] = p / p.sum()
    return probs, supports


def correct(probs, labels):
    ranked = np.argsort(-probs, axis=1, kind='stable')[:, :3]
    return np.column_stack((ranked[:, 0] == labels, (ranked == labels[:, None]).any(1))).astype(float)


def blend(neural, transitions, support, params):
    intercept, slope, entropy_slope = params
    ent_neural = -(neural * np.log(np.maximum(neural, 1e-12))).sum(1)
    ent_trans = -(transitions * np.log(np.maximum(transitions, 1e-12))).sum(1)
    raw = np.clip(intercept + slope * np.log1p(support) + entropy_slope * (ent_neural-ent_trans), -30, 30)
    transition_weight = 1 / (1 + np.exp(-raw))
    return (1-transition_weight[:, None]) * neural + transition_weight[:, None] * transitions, transition_weight


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(2)
    records = json.loads((SOURCE/'records_restricted.json').read_text())
    parts, manifest = split_records(records)
    group = parts['grouped_user']
    datasets = {part: examples(rows) for part, rows in group.items()}
    encoder = Encoder(datasets['train'])
    encoded = {part: encoder.transform(rows) for part, rows in datasets.items()}
    tables, prior = transition_tables(datasets['train'], encoder)
    trans_val, support_val = transition_predictions(datasets['val'], tables, prior)
    labels_val = encoded['val'][2].numpy()
    val_neural = []
    config = dict(CONFIG, graph_adj=graph_adjacency(encoded['train'], encoder))
    for seed in SEEDS:
        checkpoint = torch.load(SOURCE/f'grouped_user_uncp2_seed{seed}.pt', map_location='cpu', weights_only=False)
        assert checkpoint['input_vocabulary'] == encoder.inputs
        assert checkpoint['output_vocabulary'] == encoder.outputs
        model = build_model('uncp2', len(encoder.inputs)+2, len(encoder.outputs), config)
        model.load_state_dict(checkpoint['state_dict'])
        val_neural.append(predict(model, encoded['val']).astype(np.float64))
    # Small, fixed search space. Select by mean validation Top-3, then Top-1,
    # then lower NLL; the held-out test data have not been read at this point.
    choices = [(a,b,c) for a in (-4,-2,0,2,4)
               for b in (-1,-.5,0,.5,1)
               for c in (-1,0,1)]
    choices.append((-30,0,0))  # essentially pure UNCP 2
    scored = []
    for params in choices:
        values = []
        for neural in val_neural:
            mixed,_ = blend(neural, trans_val, support_val, params)
            hits = correct(mixed, labels_val)
            known = labels_val >= 0
            nll = -np.log(np.maximum(mixed[np.flatnonzero(known), labels_val[known]], 1e-12)).mean()
            values.append((hits[:,1].mean(), hits[:,0].mean(), nll))
        mean = np.mean(values, axis=0)
        scored.append((mean[0], mean[1], -mean[2], params))
    chosen = max(scored)
    params = chosen[3]
    validation_baseline_top3 = float(np.mean([correct(p,labels_val)[:,1].mean() for p in val_neural]))
    # The test set enters only after the validation choice is frozen.
    trans_test, support_test = transition_predictions(datasets['test'], tables, prior)
    labels_test = encoded['test'][2].numpy()
    runs = []
    fusion_hits = []
    baseline_hits = []
    for seed in SEEDS:
        saved = np.load(SOURCE/f'grouped_user_uncp2_seed{seed}_predictions_restricted.npz')
        assert np.array_equal(saved['targets'], labels_test)
        neural = saved['probabilities'].astype(np.float64)
        fused, weights = blend(neural, trans_test, support_test, params)
        hits = correct(fused, labels_test)
        baseline = correct(neural, labels_test)
        runs.append({'seed':seed, 'top1':float(hits[:,0].mean()),'top3':float(hits[:,1].mean()),
                     'uncp2_top1':float(baseline[:,0].mean()),'uncp2_top3':float(baseline[:,1].mean()),
                     'mean_transition_weight':float(weights.mean())})
        fusion_hits.append(hits)
        baseline_hits.append(baseline)
        np.savez_compressed(OUT/f'grouped_user_adaptive_seed{seed}.npz', correct=hits, targets=labels_test,
                            probabilities=fused.astype(np.float32))
    with (OUT/'grouped_user_adaptive_runs.csv').open('w',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=list(runs[0]));writer.writeheader();writer.writerows(runs)
    difference = np.mean(fusion_hits,axis=0) - np.mean(baseline_hits,axis=0)
    user_indices=collections.defaultdict(list)
    for index,item in enumerate(datasets['test']):
        user_indices[item['user']].append(index)
    user_sums=np.array([difference[idx].sum(0) for idx in user_indices.values()])
    user_counts=np.array([len(idx) for idx in user_indices.values()])
    draws=np.random.default_rng(20261104).integers(0,len(user_indices),(5000,len(user_indices)))
    boot=user_sums[draws].sum(1)/user_counts[draws].sum(1)[:,None]
    interval=np.quantile(boot,[.025,.975],axis=0)
    frequency_rows=[]
    for group,low,high in [('unseen_0',0,0),('rare_1_19',1,19),('medium_20_99',20,99),('frequent_100plus',100,math.inf)]:
        idx=[i for i,item in enumerate(datasets['test']) if low<=encoder.target_counts[item['target']]<=high]
        if idx:
            frequency_rows.append({'group':group,'n':len(idx),
                'uncp2_top3':float(np.mean(baseline_hits,axis=0)[idx,1].mean()),
                'fusion_top3':float(np.mean(fusion_hits,axis=0)[idx,1].mean())})
    with (OUT/'grouped_user_frequency_comparison.csv').open('w',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=list(frequency_rows[0]));writer.writeheader();writer.writerows(frequency_rows)
    report={'model':'UNCP 5','architecture':'validation-selected UNCP 2 and training-only transition fusion','protocol':'grouped_user',
            'seeds':SEEDS,'test_examples':len(labels_test),'test_users':len(manifest['grouped_user']['test']['user_ids']),
            'selected_params':dict(intercept=params[0],log_support_slope=params[1],entropy_difference_slope=params[2]),
            'validation_mean_top3':float(chosen[0]),'validation_mean_top1':float(chosen[1]),
            'validation_uncp2_mean_top3':validation_baseline_top3,
            'test_mean_top1':float(np.mean([r['top1'] for r in runs])),
            'test_mean_top3':float(np.mean([r['top3'] for r in runs])),
            'test_top1_seed_sd':float(np.std([r['top1'] for r in runs],ddof=1)),
            'test_top3_seed_sd':float(np.std([r['top3'] for r in runs],ddof=1)),
            'uncp2_mean_top1':float(np.mean([r['uncp2_top1'] for r in runs])),
            'uncp2_mean_top3':float(np.mean([r['uncp2_top3'] for r in runs])),
            'mean_transition_weight':float(np.mean([r['mean_transition_weight'] for r in runs])),
            'paired_top1_difference':float(difference[:,0].mean()),
            'paired_top3_difference':float(difference[:,1].mean()),
            'paired_user_cluster_ci95_top3':[float(interval[0,1]),float(interval[1,1])],
            'selection':'fixed 76-rule grid, validation mean Top-3 then Top-1 then NLL; test accessed after selection'}
    (OUT/'adaptive_fusion_report.json').write_text(json.dumps(report,indent=2))
    print(json.dumps(report,indent=2))




In [2]:
neural=np.array([[.15,.55,.30],[.60,.20,.20]])
transition=np.array([[.10,.70,.20],[.50,.40,.10]])
combined,weights=blend(neural,transition,np.array([5.,20.]),(2,-1,1))
assert combined.shape==(2,3) and np.allclose(combined.sum(1),1)
print('UNCP 5 adaptive fusion OK:',np.round(weights,4).tolist())

UNCP 5 adaptive fusion OK: [0.5941, 0.2616]


## Recorded experiment results

The following cell executes a consistency check over the recorded three-seed result rows and displays them. It does not retrain the model.

In [3]:
import json, numpy as np
record = json.loads("{\"summary\": {\"model\": \"UNCP 5\", \"architecture\": \"validation-selected UNCP 2 and training-only transition fusion\", \"protocol\": \"grouped_user\", \"seeds\": [11, 23, 37], \"test_examples\": 10833, \"test_users\": 11, \"selected_params\": {\"intercept\": 2, \"log_support_slope\": -1, \"entropy_difference_slope\": 1}, \"validation_mean_top3\": 0.8345820990881107, \"validation_mean_top1\": 0.6578962653191613, \"validation_uncp2_mean_top3\": 0.825056630074926, \"test_mean_top1\": 0.6670974491522816, \"test_mean_top3\": 0.8508569494445982, \"test_top1_seed_sd\": 0.008422714353512705, \"test_top3_seed_sd\": 0.002027332091452598, \"uncp2_mean_top1\": 0.646173728422413, \"uncp2_mean_top3\": 0.8357795624480753, \"mean_transition_weight\": 0.21119532680331224, \"paired_top1_difference\": 0.020923720729868613, \"paired_top3_difference\": 0.01507738699652297, \"paired_user_cluster_ci95_top3\": [0.010282369312566746, 0.017732589850673244], \"selection\": \"fixed 76-rule grid, validation mean Top-3 then Top-1 then NLL; test accessed after selection\"}, \"runs\": [{\"seed\": \"11\", \"top1\": \"0.6746053724729991\", \"top3\": \"0.8521185267238992\", \"uncp2_top1\": \"0.6569740607403305\", \"uncp2_top3\": \"0.8356872519154436\", \"mean_transition_weight\": \"0.20852521717631758\"}, {\"seed\": \"23\", \"top1\": \"0.65798947659928\", \"top3\": \"0.84851841595126\", \"uncp2_top1\": \"0.6350041539739685\", \"uncp2_top3\": \"0.8354103203175482\", \"mean_transition_weight\": \"0.21616524434512732\"}, {\"seed\": \"37\", \"top1\": \"0.6686974983845657\", \"top3\": \"0.8519339056586357\", \"uncp2_top1\": \"0.6465429705529401\", \"uncp2_top3\": \"0.8362411151112342\", \"mean_transition_weight\": \"0.20889551888849184\"}], \"behavior\": null}")
summary, runs = record['summary'], record['runs']
top1 = float(summary.get('test_mean_top1', summary.get('top1')))
top3 = float(summary.get('test_mean_top3', summary.get('top3')))
seed_top1 = np.array([float(row['top1']) for row in runs])
seed_top3 = np.array([float(row['top3']) for row in runs])
assert np.isclose(seed_top1.mean(), top1, atol=1e-9)
assert np.isclose(seed_top3.mean(), top3, atol=1e-9)
print('Recorded grouped-user experiment: 10,833 test targets, 11 held-out users')
print(f'UNCP 5  Top-1: {100*top1:.2f}%  Top-3: {100*top3:.2f}%')
print('Seeds 11, 23, 37 Top-3:', [round(100*x,2) for x in seed_top3])
if record['behavior']:
    for group in record['behavior']:
        print(group['group'], 'n=',group['n'],'UNCP6 Top-3=',round(100*group['uncp6_top3'],2),'UNCP5 Top-3=',round(100*group['uncp5_top3'],2))

Recorded grouped-user experiment: 10,833 test targets, 11 held-out users
UNCP 5  Top-1: 66.71%  Top-3: 85.09%
Seeds 11, 23, 37 Top-3: [np.float64(85.21), np.float64(84.85), np.float64(85.19)]
